## Cheat Sheet for MNLP 2026 

### Lecture 1

## Word Embedding Analogy Example

**Analogy A:** `King` − `Man` + `Woman` ≈ `?`  

**Answer:** `Queen`

We can compute this using the following line of code.  
It checks the **most similar words** using the default **cosine similarity** measure.

``` python
print(glove_vectors.most_similar(
    positive=['king', 'woman'],
    negative=['man'],
    topn=3
))
```

#### Mathematical operations with Dataframe

By selecting the interested column, we can apply min, mean or max functions

If we want to sort it by ascending order (big into small values) we can do the following: 
``` python
simlex_similar_pairs = simlex.sort_values("SimLex999", axis=0, ascending=False,).head(50) # axis = 0 sorte les lignes et axis 1 sorte les colonnes !

Where applying .head(50) allows us to select the 50 first scores
```


## Model Generation from creating Dataset to Testing the model 

If we want to create label dataset by transforming classes to binary: 
```python 
movie_reviews['target'] = movie_reviews['sentiment'].map({'positive': 1, 'negative': 0})

where map({'positive': 1, 'negative': 0}) allows us to modify the positive to 1 and negative to 0. All this are going to be stored in a new column called 'target'.

```


### Custom Dataset Creation

Now we can create a Pytorch Dataset class with the input data:

``` python
    class IMDBreviews(Dataset):
        def __init__(self, vectors, targets):
            self.x = torch.tensor(vectors, dtype=torch.float32)
            self.y = torch.tensor(targets, dtype=torch.float32)
        
        def __len__(self):
            return len(self.x)

        def __getitem__(self, idx):
            return self.x[idx], self.y[idx]


    dataset = IMDBreviews(review_vectors_max, movie_reviews['target'].values)

We split it the following way: 

    # split to train and test sets
    train, test = torch.utils.data.random_split(dataset, [0.8, 0.2])
```

### Creation of Logistic Regression 
We want to create a simple Logistic Regression function by using a class structure: 

``` python 
    # create a pytorch nn.Module class for the simple logistic regression model
    class SentimentAnalysisModel(nn.Module):

        def __init__(self,x):
            super(SentimentAnalysisModel, self).__init__()
            input_size = x.shape[1] #size of the input
            self.linear =  torch.nn.Linear(input_size, 1) #Linear(in_features, out_features) ou in_features et out_features sont la taille de input et output

        def forward(self,x):
            y_pred = torch.sigmoid(self.linear(x))
            return y_pred
        

```

### Training Structure loop
``` python
    # Training loop
    model.train()
    for epoch in range(10):
        epoch_loss = 0
        for i, data in enumerate(train_loader, 0):
            # get the inputs
            inputs, labels = data

            optimizer.zero_grad() #reset all gradients to zero before each batch
            # Forward pass: Compute predicted y by passing x to the model
            outputs = model(inputs).squeeze(1)
            
            # Compute loss
            loss = criterion(outputs, labels)
            
            
            # Perform a backward pass, and update the weights.
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        print(f'Epoch {epoch + 1} | Loss: {(epoch_loss / len(train_loader)):.4f}')
```

### Evaluation of the model - Test 

``` python
batch_loss = 0
batch_acc = 0

model.eval()
with torch.no_grad():
    for i, data in enumerate(test_loader, 0):
        # get the inputs
        input, labels = data
        

        # run forward pass
        y_pred = model(input).squeeze(1) # (64,1) ->(64,)
        
        # Compute loss
        test_loss = criterion(y_pred, labels)
        
        # Compute accuracy
        acc = BinaryAccuracy()
        acc = acc(y_pred, labels)
        
        batch_loss += test_loss.item()
        batch_acc += acc.item()

test_loss = batch_loss / len(test_loader)
test_acc = batch_acc / len(test_loader)

print(f'Test Acc: {test_acc*100:.2f}%')

```